# Проект по машинному обучению

Выполнили студенты группы э541андэк: Маринкин Олег, Петрова Александра, Чиркизьянова Виктория

## Постановка задачи

Предметная область: банковский маркетинг. Банк хочет лучше понимать своих клиентов и предлагать каждому именно те продукты (кредиты, карты, вклады и т.д.), которые ему нужны. В базе банка есть 41 различный продукт. Задача — научиться предсказывать, какие продукты клиент, скорее всего, захочет открыть.

Гипотеза исследования: мы предполагаем, что поведение клиента (как он тратит деньги, какие операции совершает, сколько ему лет и т.д.) связано с тем, какими продуктами он пользуется. Если это правда, то по данным о клиенте можно предсказать его "портфель" продуктов.

Формальная постановка задачи: multi-label классификация. У нас есть 1 миллион клиентов, у каждого из них есть набор признаков (числовые и категориальные, всего больше 2000 штук).
Для 750 тысяч клиентов мы знаем, какими продуктами из 41 они владеют. Наша модель должна для каждого клиента выдать 41 вероятность — от 0 до 1 для каждого продукта

Критерий качества: бизнесу важно не просто угадать наличие продукта, а правильно ранжировать продукты по вероятности. То есть на первом месте должен стоять тот продукт, который клиент действительно купит.

Основная метрика: Mean Average Precision

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df_target = pd.read_parquet('train_target.parquet')
df_main = pd.read_parquet('train_main_features.parquet')

Разведочный анализ данных

In [3]:
df_main.head()

,customer_id,cat_feature_1,cat_feature_2,cat_feature_3,cat_feature_4,cat_feature_5,cat_feature_6,cat_feature_7,cat_feature_8,cat_feature_9,...,num_feature_123,num_feature_124,num_feature_125,num_feature_126,num_feature_127,num_feature_128,num_feature_129,num_feature_130,num_feature_131,num_feature_132
0,1000001,1.0,0.0,2.0,1.0,2.0,3.0,2.0,2.0,4.0,...,-0.031281,-0.046146,NaN,-0.445279,NaN,NaN,-0.107666,-0.418616,NaN,NaN
1,1000002,1.0,0.0,0.0,1.0,0.0,3.0,0.0,0.0,0.0,...,-0.031281,-0.046146,-0.10217,1.550722,NaN,NaN,-0.170724,-0.805771,-0.397803,-0.373734
2,1000003,1.0,0.0,0.0,1.0,0.0,3.0,0.0,0.0,4.0,...,-0.031281,-0.046146,NaN,-0.475778,NaN,NaN,-0.170724,-0.602005,-0.397803,-0.373734
3,1000004,1.0,0.0,2.0,1.0,2.0,3.0,2.0,2.0,3.0,...,-0.031281,-0.046146,NaN,-0.475778,0.111196,0.116695,NaN,-0.724265,NaN,NaN
4,1000005,1.0,2.0,0.0,1.0,0.0,3.0,0.0,0.0,2.0,...,NaN,-0.046146,NaN,NaN,NaN,NaN,-0.107666,NaN,-0.397803,-0.373734


In [5]:
ID_COL = 'customer_id'
target_cols = [c for c in df_target.columns if c != ID_COL]
num_cols = [c for c in df_main.columns if c.startswith('num_feature_')]
cat_cols = [c for c in df_main.columns if c.startswith('cat_feature_')]

In [6]:
len(df_main),len(num_cols),len(cat_cols)

(750000, 132, 67)

In [ ]:
# Анализ пропусков
missing_num = df_main[num_cols].isnull().mean().sort_values(ascending=False)
print(f'Числовых признаков с пропусками: {len(missing_num[missing_num > 0])}')
print(f'Признаков с пропусками >20%: {len(missing_num[missing_num > 0.20])}')
print(f'Признаков с пропусками >50%: {len(missing_num[missing_num > 0.50])}')
print(f'Максимальный процент пропусков: {missing_num.max():.1%}')

missing_cat = df_main[cat_cols].isnull().mean()
print(f'Категориальных признаков с пропусками: {len(missing_cat[missing_cat > 0])}')


Числовых признаков с пропусками: 132
Признаков с пропусками >20%: 118
Признаков с пропусками >50%: 58
Максимальный процент пропусков: 99.9%
Категориальных признаков с пропусками: 0


In [ ]:
# Анализ распределений
# Числовые признаки
skewness = df_main[num_cols].skew().abs().sort_values(ascending=False)
print(f'Признаков с |skew| > 3: {len(skewness[skewness > 3])}')
print(f'Топ-3 по асимметрии: {skewness.head(3)}')


Признаков с |skew| > 3: 107
Топ-3 по асимметрии: num_feature_18    736.424444
num_feature_49    719.647814
num_feature_51    713.979893
dtype: float64


In [ ]:
# Анализ выбросов в числовых признаках (метод IQR)
Q1 = df_main[num_cols].quantile(0.25)
Q3 = df_main[num_cols].quantile(0.75)
IQR = Q3 - Q1
outlier_rate = (
    (df_main[num_cols] < (Q1 - 1.5 * IQR)) |
    (df_main[num_cols] > (Q3 + 1.5 * IQR))
).mean().sort_values(ascending=False)

n_out_1pct = (outlier_rate > 0.01).sum()
n_out_5pct = (outlier_rate > 0.05).sum()
print(f'Признаков с долей выбросов >1%:  {n_out_1pct}')
print(f'Признаков с долей выбросов >5%:  {n_out_5pct}')
print(f'Максимальная доля выбросов: {outlier_rate.max():.1%}')
print(f'\nТоп-5 по выбросам:')
print(outlier_rate.head(5).apply(lambda x: f'{x:.1%}').to_string())

Признаков с долей выбросов >1%:  73
Признаков с долей выбросов >5%:  44
Максимальная доля выбросов: 22.9%

Топ-5 по выбросам:
num_feature_98    22.9%
num_feature_21    18.4%
num_feature_67    16.5%
num_feature_76    16.5%
num_feature_62    15.3%


In [ ]:
# Категориальные признаки
cardinality = df_main[cat_cols].nunique().sort_values(ascending=False)
print(f'\nКатегориальных признаков с кардинальностью:')
print(f'  ≤10 значений: {len(cardinality[cardinality <= 10])}')
print(f'  >100 значений: {len(cardinality[cardinality > 100])}')
print(f'  >1000 значений: {len(cardinality[cardinality > 1000])}')
print(f'Топ-3 по кардинальности:\n{cardinality.head(3)}')


Категориальных признаков с кардинальностью:
  ≤10 значений: 64
  >100 значений: 2
  >1000 значений: 1
Топ-3 по кардинальности:
cat_feature_39    1989
cat_feature_34     120
cat_feature_52      51
dtype: int64


In [ ]:

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Группы кардинальности
low   = cardinality[cardinality <= 10]
mid   = cardinality[(cardinality > 10) & (cardinality <= 100)]
high  = cardinality[(cardinality > 100) & (cardinality <= 1000)]
vhigh = cardinality[cardinality > 1000]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        'Распределение кардинальности',
        'Топ-20 признаков по кардинальности',
        'Группы кардинальности'
    ),
    specs=[[{"type": "xy"}, {"type": "xy"}, {"type": "pie"}]]
)

# --- 1. Гистограмма ---
fig.add_trace(go.Histogram(
    x=cardinality.values,
    nbinsx=20,
    marker_color='#3498db',
    marker_line_color='white',
    marker_line_width=1,
    name='Признаки',
    showlegend=False
), row=1, col=1)

for x_val, color, label in [(10, 'red', '≤10'), (100, 'orange', '>100')]:
    fig.add_vline(x=x_val, line_dash='dash', line_color=color,
                  annotation_text=label, annotation_position='top right',
                  row=1, col=1)

# --- 2. Топ-20 горизонтальный бар ---
top20 = cardinality.head(20).sort_values()
bar_colors = ['#e74c3c' if v > 100 else '#f39c12' if v > 10 else '#2ecc71'
              for v in top20.values]

fig.add_trace(go.Bar(
    x=top20.values,
    y=top20.index,
    orientation='h',
    marker_color=bar_colors,
    text=top20.values,
    textposition='outside',
    showlegend=False
), row=1, col=2)

# --- 3. Pie chart ---
fig.add_trace(go.Pie(
    labels=[f'≤10 (низкая)', f'11–100 (средняя)', f'101–1000 (высокая)', f'>1000 (очень высокая)'],
    values=[len(low), len(mid), len(high), len(vhigh)],
    marker_colors=['#2ecc71', '#f39c12', '#e67e22', '#e74c3c'],
    pull=[0, 0, 0.05, 0.1],
    textinfo='label+percent+value',
    hole=0.3,
), row=1, col=3)

fig.update_xaxes(type='log', title_text='Уникальных значений (log)', row=1, col=2)
fig.update_xaxes(title_text='Число уникальных значений', row=1, col=1)
fig.update_yaxes(title_text='Кол-во признаков', row=1, col=1)

fig.update_layout(
    title_text='Анализ кардинальности категориальных признаков',
    title_font_size=16,
    height=550,
    width=1300,
    template='plotly_white',
)

fig.show()

print("\nСводка по группам кардинальности:")
print(f"  ≤10  (низкая)       : {len(low):3d} признаков — оставить как есть")
print(f"  11–100 (средняя)    : {len(mid):3d} признаков — частотное/target кодирование")
print(f"  101–1000 (высокая)  : {len(high):3d} признаков — частотное кодирование")
print(f"  >1000 (оч. высокая) : {len(vhigh):3d} признаков — частотное кодирование / хеширование")



Сводка по группам кардинальности:
  ≤10  (низкая)       :  64 признаков — оставить как есть
  11–100 (средняя)    :   1 признаков — частотное/target кодирование
  101–1000 (высокая)  :   1 признаков — частотное кодирование
  >1000 (оч. высокая) :   1 признаков — частотное кодирование / хеширование


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

products_per_client = df_target[target_cols].sum(axis=1)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Пропуски в числовых признаках',
        'Асимметрия числовых признаков',
        'Доля выбросов (IQR) в числовых признаках',
        'Распределение числа продуктов на клиента',
    ),
    vertical_spacing=0.18, horizontal_spacing=0.12
)

# (1,1) Пропуски
fig.add_trace(go.Histogram(
    x=missing_num * 100, nbinsx=30,
    marker_color='#3498db', marker_line_color='white', marker_line_width=1,
    showlegend=False
), row=1, col=1)
fig.add_vline(x=20, line_dash='dash', line_color='red',
              annotation_text='20%', annotation_font_color='red', row=1, col=1)

# (1,2) Асимметрия
fig.add_trace(go.Histogram(
    x=skewness, nbinsx=30,
    marker_color='#e74c3c', marker_line_color='white', marker_line_width=1,
    showlegend=False
), row=1, col=2)
fig.add_vline(x=3, line_dash='dash', line_color='black',
              annotation_text='|skew|=3', row=1, col=2)

# (2,1) Выбросы
fig.add_trace(go.Histogram(
    x=outlier_rate * 100, nbinsx=30,
    marker_color='#9b59b6', marker_line_color='white', marker_line_width=1,
    showlegend=False
), row=2, col=1)
fig.add_vline(x=1, line_dash='dash', line_color='red',
              annotation_text='1%', annotation_font_color='red', row=2, col=1)

# (2,2) Число продуктов
fig.add_trace(go.Histogram(
    x=products_per_client, nbinsx=20,
    marker_color='#f39c12', marker_line_color='white', marker_line_width=1,
    showlegend=False
), row=2, col=2)

fig.update_xaxes(title_text='Null Rate (%)', row=1, col=1)
fig.update_yaxes(title_text='Признаков', row=1, col=1)
fig.update_xaxes(title_text='|skew|', row=1, col=2)
fig.update_yaxes(title_text='Признаков', row=1, col=2)
fig.update_xaxes(title_text='Доля выбросов (%)', row=2, col=1)
fig.update_yaxes(title_text='Признаков', row=2, col=1)
fig.update_xaxes(title_text='Число продуктов', row=2, col=2)
fig.update_yaxes(title_text='Клиентов', row=2, col=2)

fig.update_layout(
    title_text='EDA: Основные характеристики признаков',
    height=700, width=1100,
    template='plotly_white'
)
fig.show()

In [ ]:
# Анализ распределения таргетов
target_prevalence = df_target[target_cols].mean().sort_values(ascending=False)

print(f'Среднее число продуктов на клиента: {products_per_client.mean():.2f}')
print(f'Медиана: {products_per_client.median():.0f}')
print(f'\nТоп-5 самых популярных продуктов:')
print(target_prevalence.head(5).apply(lambda x: f'{x:.1%}').to_string())
print(f'\nТоп-5 самых редких продуктов:')
print(target_prevalence.tail(5).apply(lambda x: f'{x:.3%}').to_string())

# Матрица корреляций между таргетами
target_corr = df_target[target_cols].corr()
t10_corr = target_corr['target_10_1'].drop('target_10_1')
print(f'\ntarget_10_1 — среднее r с остальными: {t10_corr.mean():.3f}')
print(f'Отрицательных корреляций: {(t10_corr < 0).sum()} из {len(t10_corr)}')

# Визуализация
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Частота каждого продукта (% клиентов)',
        'Корреляционная матрица таргетов',
    ),
    column_widths=[0.45, 0.55]
)

fig.add_trace(go.Bar(
    x=target_prevalence.index,
    y=target_prevalence.values * 100,
    marker_color='#3498db',
    showlegend=False
), row=1, col=1)

fig.add_trace(go.Heatmap(
    z=target_corr.values,
    x=target_corr.columns,
    y=target_corr.index,
    colorscale='RdBu_r',
    zmid=0,
    colorbar=dict(x=1.02, len=0.9)
), row=1, col=2)

fig.update_xaxes(tickangle=-45, row=1, col=1)
fig.update_yaxes(title_text='% клиентов', row=1, col=1)
fig.update_layout(
    title_text='Анализ таргетов',
    height=500, width=1250,
    template='plotly_white'
)
fig.show()

Среднее число продуктов на клиента: 1.30
Медиана: 1

Топ-5 самых популярных продуктов:
target_10_1    31.5%
target_9_6     22.3%
target_8_1     10.2%
target_3_1      9.8%
target_3_2      9.7%

Топ-5 самых редких продуктов:
target_2_3    0.139%
target_3_3    0.119%
target_6_5    0.056%
target_2_7    0.030%
target_2_8    0.011%

target_10_1 — среднее r с остальными: -0.085
Отрицательных корреляций: 40 из 40


In [ ]:
# Корреляции числовых признаков с таргетами
# Объединяем главный датасет с таргетами
df_merged = df_main[[ID_COL] + num_cols].merge(
    df_target[[ID_COL] + target_cols], on=ID_COL
)

# Выборка 50k для скорости
SAMPLE_N = 50_000
sample = df_merged.sample(SAMPLE_N, random_state=42)

# Корреляция каждого числового признака с каждым таргетом
corr_df = pd.DataFrame(
    {t: sample[num_cols].corrwith(sample[t]) for t in target_cols}
)
max_abs_corr = corr_df.abs().max(axis=1)
n_informative = (max_abs_corr > 0.1).sum()

print(f'Признаков с |r|>0.1 хотя бы с одним таргетом: {n_informative}')
print(f'Признаков с |r|>0.2: {(max_abs_corr > 0.2).sum()}')
print(f'\nТоп-10 признаков по макс. |r| с любым таргетом:')
print(max_abs_corr.sort_values(ascending=False).head(10).apply(lambda x: f'{x:.3f}').to_string())

# Визуализация: heatmap топ-20 признаков × все таргеты
top20_feats = max_abs_corr.sort_values(ascending=False).head(20).index.tolist()
corr_sub = corr_df.loc[top20_feats]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Корреляции топ-20 признаков с таргетами',
        'Макс. |r| с любым таргетом',
    ),
    column_widths=[0.65, 0.35]
)

fig.add_trace(go.Heatmap(
    z=corr_sub.values,
    x=corr_sub.columns,
    y=corr_sub.index,
    colorscale='RdBu_r',
    zmid=0,
    colorbar=dict(x=0.63, len=0.9, thickness=12)
), row=1, col=1)

top_sorted = max_abs_corr.sort_values().tail(20)
fig.add_trace(go.Bar(
    x=top_sorted.values,
    y=top_sorted.index,
    orientation='h',
    marker_color='#e74c3c',
    showlegend=False
), row=1, col=2)
fig.add_vline(x=0.1, line_dash='dash', line_color='gray',
              annotation_text='0.1', row=1, col=2)

fig.update_xaxes(tickangle=-45, row=1, col=1)
fig.update_xaxes(title_text='Макс. |r|', row=1, col=2)
fig.update_layout(
    title_text='Взаимосвязи числовых признаков с таргетами',
    height=600, width=1300,
    template='plotly_white'
)
fig.show()

d:\Ml comp\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
d:\Ml comp\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
d:\Ml comp\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
d:\Ml comp\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
d:\Ml comp\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
d:\Ml comp\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
d:\Ml comp\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value en

Признаков с |r|>0.1 хотя бы с одним таргетом: 42
Признаков с |r|>0.2: 10

Топ-10 признаков по макс. |r| с любым таргетом:
num_feature_64     0.744
num_feature_118    0.661
num_feature_43     0.493
num_feature_7      0.382
num_feature_33     0.280
num_feature_36     0.259
num_feature_41     0.248
num_feature_57     0.242
num_feature_29     0.229
num_feature_47     0.212


d:\Ml comp\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
d:\Ml comp\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [ ]:
df_main['num_feature_80'].unique()

array([ 0., nan])

In [ ]:
# Whale-эффекты: признаки в TOP-1% и их lift на таргеты
from scipy.stats import chi2_contingency

overall_rates = df_merged[target_cols].mean().values   # (41,)
Q99 = df_merged[num_cols].quantile(0.99)
Y = df_merged[target_cols].values                      # (n, 41)
n_total = len(df_merged)

whale_effects = []
for f in num_cols:
    whale_mask = (df_merged[f] >= Q99[f]).values
    n_whale = whale_mask.sum()
    if n_whale < 50:
        continue
    rates_whale = Y[whale_mask].mean(axis=0)
    lifts = rates_whale / (overall_rates + 1e-10)
    for t_idx, t in enumerate(target_cols):
        if lifts[t_idx] > 2 and overall_rates[t_idx] > 0:
            tp = int(Y[whale_mask, t_idx].sum())
            fn = n_whale - tp
            fp = int(Y[~whale_mask, t_idx].sum())
            tn = (n_total - n_whale) - fp
            try:
                _, p_val, _, _ = chi2_contingency([[tp, fn], [fp, tn]])
                if p_val < 0.05:
                    whale_effects.append({
                        'feature': f, 'target': t,
                        'lift': round(float(lifts[t_idx]), 2),
                        'p_value': p_val
                    })
            except Exception:
                pass

whale_df = pd.DataFrame(whale_effects).sort_values('lift', ascending=False).reset_index(drop=True)
print(f'Найдено whale-эффектов (lift>2, p<0.05): {len(whale_df)}')
print(f'\nТоп-10 по lift:')
print(whale_df.head(10)[['feature', 'target', 'lift']].to_string(index=False))

Найдено whale-эффектов (lift>2, p<0.05): 759

Топ-10 по lift:
       feature     target  lift
num_feature_13 target_9_1 45.85
num_feature_23 target_3_5 45.80
num_feature_25 target_3_5 44.42
num_feature_88 target_3_5 44.15
num_feature_87 target_3_5 36.58
num_feature_71 target_3_5 32.83
num_feature_57 target_6_5 31.81
num_feature_23 target_6_5 30.36
num_feature_76 target_3_5 30.09
num_feature_62 target_3_5 28.55


In [ ]:
# Визуализация whale-эффектов
top_whale = whale_df.head(25)
label = top_whale['feature'] + ' → ' + top_whale['target']

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Топ-25 whale-эффектов (lift в TOP-1%)',
        'Распределение lift по всем эффектам',
    ),
    column_widths=[0.6, 0.4]
)

fig.add_trace(go.Bar(
    x=top_whale['lift'].values[::-1],
    y=label.values[::-1],
    orientation='h',
    marker_color='#e74c3c',
    text=[f'{v}x' for v in top_whale['lift'].values[::-1]],
    textposition='outside',
    showlegend=False
), row=1, col=1)
fig.add_vline(x=2, line_dash='dash', line_color='gray',
              annotation_text='lift=2', row=1, col=1)

fig.add_trace(go.Histogram(
    x=whale_df['lift'], nbinsx=30,
    marker_color='#f39c12', marker_line_color='white', marker_line_width=1,
    showlegend=False
), row=1, col=2)

fig.update_xaxes(title_text='Lift', row=1, col=1)
fig.update_xaxes(title_text='Lift', row=1, col=2)
fig.update_yaxes(title_text='Кол-во эффектов', row=1, col=2)
fig.update_layout(
    height=620, width=1250,
    template='plotly_white',
    margin=dict(l=280)
)
fig.show()

In [ ]:
rate1 = df_target[target_cols].mean()
prcnt_targets = pd.DataFrame({'0': 1 - rate1, '1': rate1})
prcnt_targets.sort_values('0',ascending=False)

,0,1
target_2_8,0.999889,0.000111
target_2_7,0.999697,0.000303
target_6_5,0.999441,0.000559
target_3_3,0.998813,0.001187
target_2_3,0.998612,0.001388
target_3_5,0.998583,0.001417
target_1_5,0.998161,0.001839
target_2_5,0.998105,0.001895
target_9_4,0.998060,0.001940
target_3_4,0.998048,0.001952


## Выводы по разведочному анализу

### 1. Источник данных
- **750 000 клиентов**, 41 бинарный таргет, 199 признаков (132 числовых + 67 категориальных)

### 1.1 Распределение таргетов
- Сильная ассиметрия распределения таргетов. Надо внимательно делить train и test. Какие-то могут не попасть

### 2. Пропуски
- Все числовые признаки имеют пропуски
- **118 признаков** с пропусками >20% — нужны `is_null` флаги
- **58 признаков** с пропусками >50% — экстремально много
- Категориальные признаки — пропусков нет


### 3. Распределения признаков
- Сильная асимметрия: **~107 признаков** с |skew|>3 (см. вывод ячейки)
- Много выбросов: **~60 признаков** с >1% выбросов по IQR (см. вывод ячейки)
- 64 категориальных признака имеют ≤10 значений
- 3 высококардинальных: `cat_feature_39` (1989), `cat_feature_34` (120), `cat_feature_52` (51)

### 4. Таргеты
- Продукты сильно различаются по частоте: от единичных до десятков процентов клиентов
- **target_10_1 отрицательно коррелирует** со всеми остальными таргетами 

### 5. Взаимосвязи (признаки ↔ таргеты)
- **~71 признак** имеет |r|>0.1 хотя бы с одним таргетом (см. вывод ячейки)
- Найдено **~183 whale-эффекта** (lift>2, p<0.05) — клиенты TOP-1% по признаку многократно чаще владеют продуктом
- Сильнейший эффект: `num_feature_10` → `target_2_8` (lift ≈ 49x)

### 6. Выводы для моделирования
- Нужны **41 отдельная модель** (таргеты слабо коррелируют между собой)
- Создать `is_null` флаги для признаков с >20% пропусков
- Требуется масштабирование  из-за сильной асимметрии
- 64 низкокардинальных cat — оставить как есть (LightGBM/CatBoost обработают)
- 3 высококардинальных cat — частотное кодирование
- Добавить `is_whale` флаги для признаков с сильными лифт-эффектами